## MODELO PARA RESOLUCIÓN DE PROBLEMAS DE OPTIMIZACIÓN LINEAL CON REDES NEURONALES


### <b>Problema de minimización para el entrenamiento de Redes Neuronales</b>

Minimización de las predicciones con los valores reales (ground truth):

$$\mathcal{L} = \underbrace{||y - y_{target}||^2}_{\text{MSE}}$$

<br>
<br>

### <b>Problema de Minimización para convergencia hacia valor óptimo en Programación Lineal</b>

Minimización de función objetivo sujeto a restricciones:

- Min $C^T y$        sujeto a    $A y \geq b$

Donde:
- $C$ es el vector de costos
- $A$ es la matriz de restricciones
- $b$ es el vector de restricciones



Se define la  Función de Error como:


$$\mathcal{L} =  \underbrace{||\text{ReLU}(Ay - b)||^2}_{\text{Restricciones}} + \underbrace{(\mathbf{c}^T y)}_{\text{Función Objetivo}}$$



<br>
<br>
<br>

Por lo tanto la función de error del problema de red Neuronal, se agrega las restricciones del problema de Optimización:

$$\mathcal{L} = \underbrace{||y - y_{target}||^2}_{\text{MSE}} + \lambda_1 \underbrace{||\text{ReLU}(Ay - b)||^2}_{\text{Restricciones}} + \lambda_2 \underbrace{(\mathbf{c}^T y)}_{\text{Funcion Objetivo}}$$



Donde:
- $\lambda_1$ es el peso de la restricción
- $\lambda_2$ es el peso del objetivo


## <b>Problema de Optimización de Flujo</b>

En una red (Grapho) se busca la maximización del flujo por los enlaces que conforman dicha red tomando en cuenta las restricciones de capacidad.


Sea el Grapho $G = (N, E)$ un grapho con:
* $n$ = numero de Nodos
* $m$ = número de Enlaces
* $s$ = nodo FUENTE
* $t$ = nodo DEMANDA

* $\mathbf{f} \in \mathbb{R}^m$: Vector de Flujos
* $\mathbf{c} \in \mathbb{R}^m$: Vector de Capacidades
* $v \in \mathbb{R}$: Flujo de Demanda que sale por nodo $t$ y entra por nodo $s$.
* $A$: La matriz de incidencia $n \times m$.
* $\mathbf{d} \in \mathbb{R}^n$: Locacion de Demanda:
$$d_i = \begin{cases}
1 & \text{if } i = s \text{ (fuente)} \\
-1 & \text{if } i = t \text{ (demanda)} \\
0 & \text{otro}
\end{cases}$$

<br>
<br>

La funcion objetivo y sus restricciones se establecen como:

$$\text{maximize } v$$

Las restricciones como:
$$\begin{aligned}
& \text{Maximize} & & v \\
& \text{Subject to} & & A \mathbf{f} - v \mathbf{d} = \mathbf{0} \\
& & & \mathbf{0} \le \mathbf{f} \le \mathbf{c}
\end{aligned}$$


## <b>Planteameniento del Problema Híbrido</b>


$$\mathcal{L} = \underbrace{||y - y_{target}||^2}_{\text{MSE}} + \lambda_1 \underbrace{||\text{ReLU}(Ay)||^2}_{\text{Conservacion de Flujo}}+ \lambda_1 \underbrace{||\text{ReLU}(-y + c)||^2}_{\text{Limite de Capacidad}} + \lambda_2 \underbrace{(\mathbf{c}^T y)}_{\text{Funcion Objetivo}} + $$



In [ ]:
import networkx as nx
import numpy as np
import pandas as pd

from IPython.display import clear_output

import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
G = nx.DiGraph()
G.add_edge('INPUT', 'HR_A', capacity=10000)
G.add_edge('INPUT', 'HR_B', capacity=10000)
G.add_edge('HR_A', 'MR_A', capacity=10000)
G.add_edge('HR_B', 'MR_B', capacity=10000)
G.add_edge('MR_B', 'MR_A', capacity=10000)
G.add_edge('MR_A', 'C', capacity=10000)

G.add_edge('C', 'D', capacity=10000)
G.add_edge('MR_B', 'A', capacity=10000)
G.add_edge('A', 'B', capacity=10000)
G.add_edge('B', 'D', capacity=10000)
G.add_edge('D', 'E', capacity=10000)

G.add_edge('C', 'S7', capacity=10000)
G.add_edge('D', 'S6', capacity=10000)
G.add_edge('D', 'S5', capacity=10000)
G.add_edge('B', 'S3', capacity=10000)
G.add_edge('A', 'S1', capacity=10000)
G.add_edge('A', 'S2', capacity=10000)
G.add_edge('E', 'S4', capacity=10000)

G.add_edge('S1', 'OUTPUT', capacity=10000)
G.add_edge('S2', 'OUTPUT', capacity=10000)
G.add_edge('S3', 'OUTPUT', capacity=10000)
G.add_edge('S4', 'OUTPUT', capacity=10000)
G.add_edge('S5', 'OUTPUT', capacity=10000)
G.add_edge('S6', 'OUTPUT', capacity=10000)
G.add_edge('S7', 'OUTPUT', capacity=10000)


NODES_ID = {x : i for i, x in enumerate(list(G.nodes()))}
EDGES = [(NODES_ID[x[0]], NODES_ID[x[1]], x[2]['capacity']) for x in G.edges(data=True)]

print(NODES_ID)
print(EDGES)

{'INPUT': 0, 'HR_A': 1, 'HR_B': 2, 'MR_A': 3, 'MR_B': 4, 'C': 5, 'D': 6, 'A': 7, 'B': 8, 'E': 9, 'S7': 10, 'S6': 11, 'S5': 12, 'S3': 13, 'S1': 14, 'S2': 15, 'S4': 16, 'OUTPUT': 17}
[(0, 1, 10000), (0, 2, 10000), (1, 3, 10000), (2, 4, 10000), (3, 5, 10000), (4, 3, 10000), (4, 7, 10000), (5, 6, 10000), (5, 10, 10000), (6, 9, 10000), (6, 11, 10000), (6, 12, 10000), (7, 8, 10000), (7, 14, 10000), (7, 15, 10000), (8, 6, 10000), (8, 13, 10000), (9, 16, 10000), (10, 17, 10000), (11, 17, 10000), (12, 17, 10000), (13, 17, 10000), (14, 17, 10000), (15, 17, 10000), (16, 17, 10000)]


In [ ]:
NUM_NODES = G.number_of_nodes()
SOURCE_NODE = 0
SINK_NODE = 17

# Hyperparameters
LEARNING_RATE = 0.1
EPOCHS = 100
LAMBDA_PHYSICS = 100.0


print(f"Number of nodes: {NUM_NODES}")
print(f"Number of edges: {len(EDGES)}")
EDGES

Number of nodes: 18
Number of edges: 25


[(0, 1, 10000),
 (0, 2, 10000),
 (1, 3, 10000),
 (2, 4, 10000),
 (3, 5, 10000),
 (4, 3, 10000),
 (4, 7, 10000),
 (5, 6, 10000),
 (5, 10, 10000),
 (6, 9, 10000),
 (6, 11, 10000),
 (6, 12, 10000),
 (7, 8, 10000),
 (7, 14, 10000),
 (7, 15, 10000),
 (8, 6, 10000),
 (8, 13, 10000),
 (9, 16, 10000),
 (10, 17, 10000),
 (11, 17, 10000),
 (12, 17, 10000),
 (13, 17, 10000),
 (14, 17, 10000),
 (15, 17, 10000),
 (16, 17, 10000)]

In [ ]:
def build_incidence_matrix(edges, num_nodes):
    indices = []
    values = []
    for i, (u, v, _) in enumerate(edges):
        # Column indices for the two directions of this edge
        idx_fwd, idx_bwd = 2 * i, 2 * i + 1

        # u -> v (u loses, v gains)
        indices.extend([[u, idx_fwd], [v, idx_fwd]])
        values.extend([-1.0, 1.0])

        # v -> u (v loses, u gains)
        indices.extend([[v, idx_bwd], [u, idx_bwd]])
        values.extend([-1.0, 1.0])

    i = torch.LongTensor(indices).t()
    v = torch.FloatTensor(values)
    return torch.sparse_coo_tensor(i, v, size=(num_nodes, 2 * len(edges)))

def vectorized_flow_loss(flows, incidence_matrix, capacities, source, sink, lambda_phys):
    # 1. Calculate Net Flow for all nodes (Matrix Mult)
    # Flatten flows to (2*E, 1) for multiplication
    flat_flows = flows.view(-1).unsqueeze(1)
    net_flows = torch.sparse.mm(incidence_matrix, flat_flows).squeeze()

    # 2. Conservation Loss (Nodes must balance to 0, except Source/Sink)
    mask = torch.ones(net_flows.shape[0], dtype=torch.bool)
    mask[source] = False
    mask[sink] = False
    loss_cons = torch.sum(net_flows[mask] ** 2)

    # 3. Capacity Loss (Sum of both dirs must be <= Capacity)
    # Sum flow in both directions: shape (E, 2) -> (E,)
    total_edge_flow = torch.sum(flows, dim=1)
    loss_cap = torch.sum(torch.relu(total_edge_flow - capacities) ** 2)

    # 4. Objective: Maximize Flow into Sink
    # We minimize negative flow
    flow_at_sink = net_flows[sink]
    loss_obj = -flow_at_sink

    return (lambda_phys * loss_cons) + (lambda_phys * loss_cap) + loss_obj, flow_at_sink

In [ ]:
class FlowNetwork(nn.Module):
    def __init__(self, num_edges):
        super().__init__()
        # Parameters: Raw numbers that can be negative
        self.raw_weights = nn.Parameter(torch.randn(num_edges, 2))

    def forward(self):
        # Apply ReLU so the solver physically cannot output negative flow
        return torch.relu(self.raw_weights)

In [ ]:
incidence_matrix = build_incidence_matrix(EDGES, NUM_NODES)
capacities = torch.tensor([e[2] for e in EDGES], dtype=torch.float32)

model = FlowNetwork(len(EDGES))
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)


In [ ]:

print(f"{'Epoch':<10} | {'Loss':<10} | {'Sink Flow':<10} | {'Status'}")
print("-" * 50)

for epoch in range(EPOCHS):
    optimizer.zero_grad()  # Reset gradients

    current_flows = model()


    loss, sink_flow = vectorized_flow_loss(
        current_flows, incidence_matrix, capacities,
        SOURCE_NODE, SINK_NODE, LAMBDA_PHYSICS
    )


    loss.backward()
    optimizer.step()

    if epoch % 500 == 0 or epoch == EPOCHS - 1:
        print(f"{epoch:<10} | {loss.item():.4f}     | {sink_flow.item():.4f}     | Training...")


print("\n--- FINAL RESULTS ---")
final_flows = model().detach()
total_sink_flow = 0

for i, (u, v, cap) in enumerate(EDGES):
    f_uv = final_flows[i, 0].item()
    f_vu = final_flows[i, 1].item()

    # Only print significant flows
    if f_uv > 0.01 or f_vu > 0.01:
        print(f"Edge {u}-{v} (Max {cap}): \t{f_uv:.2f} -> \t{f_vu:.2f} <-")

print(f"\nFinal Calculated Max Flow: {vectorized_flow_loss(final_flows, incidence_matrix, capacities, SOURCE_NODE, SINK_NODE, 0)[1].item():.2f}")


Epoch      | Loss       | Sink Flow  | Status
--------------------------------------------------
0          | -0.0120     | 0.0287     | Training...
99         | -0.0175     | 0.0349     | Training...

--- FINAL RESULTS ---
Edge 1-3 (Max 10000): 	0.33 -> 	0.33 <-
Edge 2-4 (Max 10000): 	0.87 -> 	0.87 <-
Edge 5-6 (Max 10000): 	0.78 -> 	0.77 <-
Edge 5-10 (Max 10000): 	0.90 -> 	0.91 <-
Edge 6-9 (Max 10000): 	0.00 -> 	0.61 <-
Edge 8-6 (Max 10000): 	0.00 -> 	0.62 <-
Edge 8-13 (Max 10000): 	0.63 -> 	0.00 <-
Edge 9-16 (Max 10000): 	0.00 -> 	0.60 <-
Edge 13-17 (Max 10000): 	0.63 -> 	0.00 <-
Edge 16-17 (Max 10000): 	0.00 -> 	0.60 <-

Final Calculated Max Flow: 0.03
